# VTuber Builder · Colab 기본 화면 (Gradio 없음)


## ① 환경 설치 · 실행 완료 후 셀 종료


In [ ]:
import os, pathlib, shutil, signal, subprocess, sys, time

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
SETUP_LOG = pathlib.Path("/content/vtuber_builder/logs/colab_bootstrap.log")

def run(command, timeout):
    """Log externally; interrupting Colab stops the entire install process group."""
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    print("[준비]", " ".join(map(str, command[:4])), "(전체 로그:", SETUP_LOG, ")", flush=True)
    with SETUP_LOG.open("a", encoding="utf-8") as output:
        output.write("\n$ " + subprocess.list2cmdline(command) + "\n")
        output.flush()
        process = subprocess.Popen(
            command, stdout=output, stderr=subprocess.STDOUT,
            start_new_session=True,
        )
        started = last_report = time.monotonic()
        try:
            while process.poll() is None:
                now = time.monotonic()
                if now - started > timeout:
                    raise TimeoutError(f"환경 준비 제한 시간 {timeout}초 초과")
                if now - last_report >= 20:
                    print("[준비 진행 중]", int(now-started), "초 · 중단하면 하위 작업도 종료", flush=True)
                    last_report = now
                time.sleep(0.5)
        except BaseException:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
                process.wait(timeout=10)
            print("환경 준비 작업 종료 요청 완료", flush=True)
            raise
    if process.returncode:
        error_tail = "".join(SETUP_LOG.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-80:])
        raise RuntimeError(
            f"환경 준비 실패 (exit={process.returncode})\n{error_tail}\n전체 로그: {SETUP_LOG}"
        )

print("① 저장소 동기화")
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)
run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)

sys.path.insert(0, str(REPO_DIR))
from tools.colab_native import stop_legacy_server
stop_legacy_server()

print("① Python / CUDA 환경 확인")
setup_code = (
    "import os,pathlib,runpy; os.environ['VTUBER_SETUP_ONLY']='1'; "
    "repo=pathlib.Path('/content/Virtual-pipeline'); "
    "app=runpy.run_path(str(repo/'tools'/'colab_app.py'),run_name='vtuber_prepare'); "
    "head,logs=app['ensure_runtime'](); "
    "print('준비 완료:',head[:12],flush=True)"
)
run([sys.executable, "-u", "-c", setup_code], 9000)
print("① 준비 완료. 이제 ②에서 모드를 선택하세요.")


## ② 생성 모드 및 용도 선택


In [ ]:
#@title ② 생성 설정 · Colab 기본 입력 화면
MODE = "3d" #@param ["inochi2d", "live2d", "3d", "accessory"]
USAGE = "corporation" #@param ["corporation", "personalProfit", "personalNonProfit"]
EXISTING_IMAGE_PATH = "" #@param {type:"string"}
MULTI_REFERENCE_3D = False #@param {type:"boolean"}
ACCESSORY_ANCHOR = "HEAD_TOP" #@param ["HEAD_TOP", "FACE", "LEFT_EAR", "RIGHT_EAR", "NECK", "CHEST", "BACK", "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HAND", "RIGHT_HAND", "LEFT_FOOT", "RIGHT_FOOT", "HIPS"]
print(f"선택: {MODE} · 용도: {USAGE}")
print("이미지 경로를 비워 두면 ③ 셀에서 Colab 기본 파일 업로드 창이 열립니다.")


## ③ 캐릭터 사진 업로드 및 생성 (중단 가능)


In [ ]:
#@title ③ 캐릭터/액세서리 생성 · 이 셀을 중단하면 AI 작업도 종료됩니다
import pathlib, sys

repo = pathlib.Path("/content/Virtual-pipeline")
if not (repo / "tools" / "colab_native.py").is_file():
    raise RuntimeError("① 저장소 동기화 셀부터 먼저 실행하세요.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
from tools.colab_native import generate

RESULT_FILE = generate(
    MODE, USAGE,
    image_path=EXISTING_IMAGE_PATH.strip(),
    full_body=MULTI_REFERENCE_3D,
    accessory_anchor=ACCESSORY_ANCHOR,
)
if RESULT_FILE:
    print("완료 파일:", RESULT_FILE)
    print("브라우저 저장이 필요하면 아래 코드를 새 셀에서 실행하세요:")
    print("from google.colab import files; files.download(RESULT_FILE)")


결과 파일은 Colab 왼쪽 파일 탐색기에서 찾을 수 있습니다. 다운로드는 필요할 때만 별도 셀에서 `from google.colab import files; files.download(RESULT_FILE)`을 실행하세요.
